# Label lab: specimen label table

Reads the label fields (individual ID, species, age, sex, feather line, date) from the **original** slides,
scores them against hand-checked answers, and shows the exact crops the model saw.

**Two ways to use it**
- **Inspect a finished run**: set `RUN_DIR` to an existing folder under `data/label_runs/` and `RUN_NEW = False`.
- **Run a new sample**: set `RUN_NEW = True`, choose the slides (`SELECT`), and choose where it runs:
  `MODE = "local"` reads from this workspace through the VLM server; `MODE = "distributed"` sends one
  `extract_labels` task per slide to the cluster workers (needs the label code deployed on them).
  A run folder can be re-run: finished slides are skipped and slides with a `model_error` are retried.

**Fields and statuses.** `file_id` is the file name (e.g. `A0012 1996-im1058.jpg`); `individual_id` is the code
printed on the label (e.g. `A0012H`). Each field tries its primary crop, an expanded crop, then wider fallback crops.
`readable` · `partial` (some text hidden) · `not_found` (absent in every crop, exported `N/A`) ·
`needs_review` (bad format, or two readings disagree) · `conflict` (disagrees with the file name) ·
`filename_fallback` (nothing on the slide, value from the file name) · `model_error` (request failed; **not** the
same as missing, exported blank).

Regions, prompts and format rules: `configs/label_fields.json` (re-read on every run, no restart).
Kernel: **bird-feather-analysis (pixi)**.

## 1. Parameters

In [ ]:
RUN_NEW = False                       # True: run SELECT below into a new folder; False: just inspect RUN_DIR
RUN_DIR = "../data/label_runs/sample30-v2"   # folder to inspect (or to resume, with RUN_NEW = True)

SELECT = "manifest"                   # "manifest" | "stratified" | "random" | "match"
MANIFEST = "../configs/label_sample_30.txt"
N, SEED = 10, 1                       # for "stratified" / "random"
MATCH = ["A0012 1996"]                # for "match": substrings of file names
MODE = "local"                        # "local" (this workspace + VLM server) | "distributed" (cluster workers)
FIELDS = None                         # e.g. ["date", "individual_id"] to iterate on a few fields; None = all
WORKERS = 2                           # slides in flight at once (local mode)

CONFIG = "../configs/label_fields.json"
GOLD = "../data/label_gold.csv"       # hand-checked answers (blank cell = not scored, N/A = truly absent)

## 2. Setup

In [ ]:
import json, random, sys, time
from pathlib import Path
sys.path.insert(0, "..")
import pandas as pd
from IPython.display import display, Markdown
from PIL import Image, ImageDraw
from src import label_extract as lx
from src.notebook_remote_runtime import RemoteClusterIO, load_notebook_config_from_env

pd.set_option("display.max_colwidth", 80)
rc = load_notebook_config_from_env()
remote = RemoteClusterIO(rc)
CACHE = Path("../data/raw_cache"); CACHE.mkdir(parents=True, exist_ok=True)

def local_copy(path_or_name):
    """The slide in data/raw_cache, fetched from the head node if needed."""
    p = CACHE / Path(path_or_name).name
    if not p.exists():
        remote.fetch_remote_file(rc.head_host, remote.remote_runtime_path(f"{rc.remote_input_dir}/{p.name}"), CACHE)
    return p

## 3. Run a new sample (only when `RUN_NEW = True`)

In [ ]:
if RUN_NEW:
    cfg = lx.load_config(CONFIG)
    all_images = remote.list_remote_images()
    by_name = {Path(p).name: p for p in all_images}
    if SELECT == "manifest":
        names = [l.strip() for l in Path(MANIFEST).read_text().splitlines() if l.strip() and not l.startswith("#")]
        chosen = [by_name[n] for n in names if n in by_name]
    elif SELECT == "stratified":
        chosen = lx.stratified_sample(all_images, N, SEED)
    elif SELECT == "match":
        chosen = [p for p in all_images if any(m in Path(p).name for m in MATCH)]
    else:
        chosen = random.Random(SEED).sample(all_images, min(N, len(all_images)))
    out = Path(RUN_DIR) if RUN_DIR and Path(RUN_DIR).exists() else Path("../data/label_runs") / time.strftime("%Y%m%d-%H%M%S")
    out.mkdir(parents=True, exist_ok=True)
    (out / "config_used.json").write_text(json.dumps(cfg, indent=2))
    (out / "manifest.txt").write_text("\n".join(Path(p).name for p in chosen) + "\n")
    todo = lx.pending_images(chosen, out)
    print(f"{len(chosen)} slides -> {out}  ({len(chosen) - len(todo)} already done, {len(todo)} to run, mode={MODE})")
    t0 = time.time()
    if MODE == "distributed":
        lx.run_distributed(todo, out, log=lambda s: print(s, flush=True))
    else:
        lx.run_local(todo, out, cfg, FIELDS, WORKERS, log=lambda s: print(s, flush=True))
    lx.finish_run(out, chosen, cfg, GOLD)
    print(f"done in {time.time() - t0:.0f}s")
    RUN_DIR = str(out)
else:
    print("RUN_NEW = False: inspecting", RUN_DIR)

## 4. Load the run

In [ ]:
run = Path(RUN_DIR)
cfg = json.loads((run / "config_used.json").read_text()) if (run / "config_used.json").exists() else lx.load_config(CONFIG)
latest = lx.load_previous(run / "labels.jsonl")
order = ([l.strip() for l in (run / "manifest.txt").read_text().splitlines() if l.strip()]
         if (run / "manifest.txt").exists() else list(latest))
results = [latest[n] for n in order if n in latest]
summary = lx.finish_run(run, [r.image for r in results], cfg, GOLD)   # refresh exports + scores
print(f"{run}: {summary['slides']} slides, {summary['needs_review']} need review, "
      f"{summary['slides_with_model_error']} with model errors, {summary['mean_seconds_per_slide']} s/slide")
status = pd.DataFrame(summary["status_counts"]).fillna(0).astype(int).T
display(status)

## 5. Label table
Colours: green readable · yellow partial / file-name fallback · grey not found · red needs review / conflict · dark red model error.

In [ ]:
STATUS_COLORS = {"readable": "#d4edda", "partial": "#fff3cd", "not_found": "#e2e3e5",
                 "filename_fallback": "#fff3cd", "needs_review": "#f8d7da", "conflict": "#f8d7da",
                 "model_error": "#f5c6cb"}
df = pd.DataFrame([r.row() for r in results])
value_cols = ["file_id", "individual_id", "species", "age", "sex", "feather", "year", "month", "day"]
def _color(col):
    st = df.get(col + "_status")
    if col in ("year", "month", "day"):
        st = df.get("date_status")
    return [f"background-color: {STATUS_COLORS.get(s, '')}" for s in st] if st is not None else [""] * len(df)
display(df[value_cols].style.apply(lambda c: _color(c.name) if c.name != "file_id" else [""] * len(c)))

## 6. Scores against the hand-checked answers
Exact match for the coded fields. `wrong_value` = a wrong value accepted as readable; `false_na` = text was there but the run said not found.

In [ ]:
if "gold" in summary:
    exact = pd.DataFrame({k: {kk: v[kk] for kk in ("n", "correct", "accuracy", "wrong_value", "false_na")}
                          for k, v in summary["gold"].items()}).T
    display(exact)
    misses = [dict(field=k, file_id=w[0], got=w[1], want=w[2], status=w[3])
              for k, v in summary["gold"].items() for w in v["wrong"]]
    if misses:
        display(Markdown("**Misses**")); display(pd.DataFrame(misses))
else:
    print("no gold file at", GOLD)

### Feather line (free text), scored separately
`visibility`: full / partial / none against the answer file; `CER` = character error rate on the visible text (0 = exact); `invented` = text returned where none is visible; `overreach` = more than 2 characters beyond what is visible on a partly hidden line (completing hidden text).

In [ ]:
if "gold_feather_text" in summary:
    t = summary["gold_feather_text"]
    display(pd.DataFrame([{k: t[k] for k in ("n", "visibility_accuracy", "text_vs_none_accuracy", "with_text",
                                             "mean_cer", "exact", "invented_text", "overreach", "model_error")}]))
    display(Markdown("**Visibility (answer -> run)**")); display(pd.Series(t["confusion"]).sort_index().to_frame("slides"))
    display(pd.DataFrame(t["rows"]))

## 7. Review gallery
Every field that is not plainly readable/not found, with the crops the model actually saw (evidence saved by the run).

In [ ]:
REVIEW = ("partial", "needs_review", "conflict", "filename_fallback", "model_error")
def evidence_images(r, field):
    """Saved evidence crops for one field; recomputed from the slide when the run kept them on a worker."""
    paths = [Path(a.evidence) for a in r.fields[field].attempts if a.evidence]
    local = [p for p in paths if p.exists()]
    if local:
        return [(p.stem, Image.open(p)) for p in local]
    im = Image.open(local_copy(r.image))
    return [(f"{field}_{name}", lx.crop_field(im, reg, 900)) for name, reg in lx.crop_plan(cfg["fields"][field], cfg)]

for r in results:
    bad = {n: f for n, f in r.fields.items() if f.status in REVIEW}
    if not bad:
        continue
    display(Markdown(f"---\n### {r.file_id}"))
    for n, f in bad.items():
        print(f"{n}: {f.value!r}  [{f.status}] via {f.source or '-'}  {f.note}")
        for name, im in evidence_images(r, n)[:4]:
            im = im.copy(); im.thumbnail((640, 200)); print("  ", name); display(im)

## 8. One slide, all regions
The slide with every field's crop plan drawn (solid = primary, dashed-looking thin = expanded/fallback) and what the run read.

In [ ]:
SLIDE = results[0].file_id if results else "A0012 1996-im1058.jpg"   # any file_id from the table
r = latest.get(SLIDE)
im = Image.open(local_copy(SLIDE)).convert("RGB"); w, h = im.size
d = ImageDraw.Draw(im)
colors = {"individual_id": "red", "species": "orange", "age": "purple", "sex": "blue", "feather": "green", "date": "magenta"}
for n, spec in cfg["fields"].items():
    for i, (name, (l, t, rr, b)) in enumerate(lx.crop_plan(spec, cfg)):
        d.rectangle([l*w, t*h, rr*w, b*h], outline=colors.get(n, "red"), width=10 if i == 0 else 3)
    l, t, _, _ = spec["region"]; d.text((l*w + 14, t*h + 14), n, fill=colors.get(n, "red"))
im.thumbnail((900, 900)); display(im)
if r:
    display(pd.DataFrame([{"field": n, "value": f.value, "status": f.status, "source": f.source, "note": f.note}
                          for n, f in r.fields.items()]))

## 9. Outputs
`labels.csv` (flat table, `N/A` only for not found), `review.csv`, `labels.jsonl` (everything, including every crop attempt and raw model answer), `labels.sqlite3` (`specimen_labels`), `summary.json`, `evidence/`.

CLI equivalents: `pixi run labels -- --manifest configs/label_sample_30.txt --gold data/label_gold.csv` (add `--distributed` for the workers, `--out <dir>` to resume).

In [ ]:
for p in sorted(run.iterdir()):
    print(f"{p.name:22} {'dir' if p.is_dir() else f'{p.stat().st_size/1024:.0f} KB'}")